In [4]:
import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [6]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-02-20 17:29:20--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-21T00%3A24%3A48Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-20T23%3A24%3A21Z&ske=2026-02-21T00%3A24%3A48Z&sks=b&skv=2018-11-09&sig=9DkZ3rnLhfo6vi0xdAvE0PRq7ilna2kTzmCwkKJbJwM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MTYzMzc2MCwibmJmIjoxNzcxNjMwMTYwLCJwYXRoIjoi

In [7]:
!gzip -d fhvhv_tripdata_2021-01.csv.gz

In [8]:
# Check how many rows Spark will have to process
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [9]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [12]:
import pandas as pd

In [13]:
df_pandas = pd.read_csv('head.csv')

In [16]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [19]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [20]:
from pyspark.sql import types

In [21]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [23]:
df = spark.read \
    .option('header','true') \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [24]:
df = df.repartition(24)

In [25]:
df.write.parquet('fhvhv/2021/01/')

26/02/20 17:40:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/20 17:40:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/20 17:40:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/20 17:40:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/20 17:40:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/20 17:40:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/20 17:40:11 WARN MemoryManager: Total allocation exceeds 95.00%

In [26]:
df = spark.read.parquet('fhvhv/2021/01/')

In [27]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [30]:
from pyspark.sql import functions as F

In [40]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+------------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|dispatch_id_native|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+------------------+
|           HV0003|              B02617|2021-01-04 13:55:03|2021-01-04 14:00:50|          21|         178|   NULL|             e/a39|
|           HV0003|              B02879|2021-01-02 13:44:12|2021-01-02 14:13:35|         121|         244|   NULL|             e/b3f|
|           HV0003|              B02875|2021-01-01 02:17:09|2021-01-01 02:32:39|          37|          61|   NULL|             e/b3b|
|           HV0003|              B02877|2021-01-01 04:12:19|2021-01-01 04:23:04|         157|         226|   NULL|             s/b3d|
|           HV0003|              B02836|2021-01-01 03:18:28|20

In [42]:
num_col = F.substring(F.col("dispatching_base_num"), 2, 10).cast("int")

# Step 2: Prepare the hex string (lowercase and padded to 3 chars)
hex_str = F.lower(F.lpad(F.hex(num_col), 3, '0'))

# 2. Run the transformation (Note: we don't put df = ... if we are just showing)
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn("dispatch_id_native", 
        F.when(num_col % 7 == 0, F.concat(F.lit("s/"), hex_str))
         .when(num_col % 3 == 0, F.concat(F.lit("a/"), hex_str))
         .otherwise(F.concat(F.lit("e/"), hex_str))) \
    .select(
        'dispatching_base_num',  # Original name
        'dispatch_id_native',    # Your new "crazy" hex column
        'pickup_date', 
        'dropoff_date', 
        'PULocationID', 
        'DOLocationID'
    ) \
    .show(5)

+--------------------+------------------+-----------+------------+------------+------------+
|dispatching_base_num|dispatch_id_native|pickup_date|dropoff_date|PULocationID|DOLocationID|
+--------------------+------------------+-----------+------------+------------+------------+
|              B02617|             e/a39| 2021-01-04|  2021-01-04|          21|         178|
|              B02879|             e/b3f| 2021-01-02|  2021-01-02|         121|         244|
|              B02875|             e/b3b| 2021-01-01|  2021-01-01|          37|          61|
|              B02877|             s/b3d| 2021-01-01|  2021-01-01|         157|         226|
|              B02836|             e/b14| 2021-01-01|  2021-01-01|         192|         134|
+--------------------+------------------+-----------+------------+------------+------------+
only showing top 5 rows


In [43]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [44]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


26/02/20 18:52:07 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 482127 ms exceeds timeout 120000 ms
26/02/20 18:52:07 WARN SparkContext: Killing executors is not supported by current scheduler.
26/02/20 19:10:06 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$